# The Environment Class:

In [2]:
class Board:
    def __init__(self):
        self.state = [0] * 25

    def show_state(self, state):
      row = ''
      for i in range(1, 26):
          if i % 5 == 1:
              row = '| '
          if state[i - 1] == 1:
              row += 'X | '
          elif state[i - 1] == -1:
              row += 'O | '
          else:
              row += '  | '
          if i % 5 == 0:
              print(row)


    def get_state(self):
        return self.state

    def update_state(self, player, action):
        """
        player = 1 or -1,
        action = index of the cell of the board that is about to be played
        """
        self.state[action] = player

    @staticmethod
    def full_board(state):
        if not (0 in state):
            return True
        return False


In [4]:
import numpy as np
import random
import hashlib
import math

# The Agent Class where the Q_table is made and updated:

In [15]:
class Agent:
    def __init__(self, player, epsilon, gamma, learning_rate):
        self.player = player
        self.exploration_rate = epsilon
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.possible_moves = list(range(0, 25))
        self.unique_hashed_states = []
        self.unique_states = []
        self.board = Board()
        self.q_table = {}

    def get_q_value(self, state_hash, action):
        return self.q_table.get((state_hash, action), 0)

    def update_q_value(self, state_hash, action, new_value):
        self.q_table[(state_hash, action)] = new_value

    def move(self, cell):
        self.board.update_state(self.player, cell)
        self.possible_moves.remove(cell)

    def game_over(self, state):
        opponent = self.player * -1
        # checking rows
        for i in range(0, 25, 5):
            if sum(state[i:i + 5]) == opponent * 5:
                return True
        # checking columns
        for i in range(0, 5):
            if sum(state[i:25:5]) == opponent * 5:
                return True
        # checking diagonals
        if sum(state[0:25:6]) == opponent * 5:
            return True
        if sum(state[4:21:4]) == opponent * 5:
            return True
        return False

    def win(self, state):
        # checking rows
        for i in range(0, 25, 5):
            if sum(state[i:i + 5]) == self.player * 5:
                return True
        # checking columns
        for i in range(0, 5):
            if sum(state[i:25:5]) == self.player * 5:
                return True
        # checking for diagonals
        if sum(state[0:25:6]) == self.player * 5:
            return True
        if sum(state[4:21:4]) == self.player * 5:
            return True
        return False

    def tie(self, state):
        if Board.full_board(state):
            if not self.win(state) and not self.game_over(state):
                return True
            return False
        return False

    def reward(self, state):
        if self.win(state):
            return 5
        if self.game_over(state):
            return -5
        if self.tie(state):
            return -0.2
        # to discourage unnecessary moves
        return -0.1

    def canonical_state_hash(self, state):
        """
        Finds the canonical form of the board state, and returns its hash and
        the canonical state itself in the same format as the input `state`.
        """
        # Converting state to a 5x5 numpy array
        board = np.array(state).reshape(5, 5)

        # Defining all possible transformations
        transformations = [
            board,
            np.rot90(board, 1),  # 90 degrees rotation
            np.rot90(board, 2),  # 180 degrees rotation
            np.rot90(board, 3),  # 270 degrees rotation
            np.fliplr(board),  # Horizontal reflection
            np.flipud(board),  # Vertical reflection
        ]

        # Calculating hash for each transformation and finding the minimum
        min_hashed_state = None
        canonical_index = 0
        canonical_state = state

        for i, transformation in enumerate(transformations):
            transformation_hash = hashlib.md5(transformation.flatten().tobytes()).hexdigest()
            if min_hashed_state is None or transformation_hash < min_hashed_state:
                min_hashed_state = transformation_hash
                canonical_index = i
                # Storing canonical state as 1D list
                canonical_state = transformation.flatten().tolist()

        return min_hashed_state, canonical_state, canonical_index

    def inverse_canonical_move(self, canonical_move, transformation_index):
        """
        Maps a move in the canonical state back to the original state using
        the inverse of the transformation indicated by transformation_index.
        """
        # Converting the move to 2D coordinates (row, col)
        row, col = divmod(canonical_move, 5)

        # Initializing a blank 5x5 board with the move marked in the canonical form
        canonical_board = np.zeros((5, 5), dtype=int)
        # Marking the move position
        canonical_board[row, col] = 1

        # Applying the inverse of the transformation
        if transformation_index == 1:  # 90 degrees rotation
            original_board = np.rot90(canonical_board, -1)
        elif transformation_index == 2:  # 180 degrees rotation
            original_board = np.rot90(canonical_board, -2)
        elif transformation_index == 3:  # 270 degrees rotation
            original_board = np.rot90(canonical_board, -3)
        elif transformation_index == 4:  # Horizontal reflection
            original_board = np.fliplr(canonical_board)
        elif transformation_index == 5:  # Vertical reflection
            original_board = np.flipud(canonical_board)
        else:
            original_board = canonical_board  # No transformation applied

        # Finding the original move location in the transformed original board
        original_move = np.argmax(original_board.flatten())
        return original_move

    def update_q_table(self, hashed_canonical_current_state, canonical_move, reward, canonical_next_state,
                       hashed_canonical_next_state):
        """
                Update the Q-value for a given (state, action) pair based on the reward and
                the estimated future rewards from the next state.

                Parameters:
                - hashed_canonical_current_state: The hash of the current state when transformed to its canonical form.
                - canonical_move: The action taken in the current state.
                - reward: The immediate reward received after taking the action.
                - canonical_next_state: The state reached after taking the action transformed into its canonical form.
                - hashed_canonical_next_state: The hash of the state reached after taking the action when transformed into its canonical form.
        """

        current_q_value = self.get_q_value(hashed_canonical_current_state, canonical_move)
        if self.is_terminal(canonical_next_state):  # Assuming is_terminal checks for win/loss/draw
          max_next_q_value = 0
        else:
          _, max_next_q_value = self.max_q_value(canonical_next_state, hashed_canonical_next_state)
        updated_q_value = current_q_value + self.learning_rate * (
                    reward + self.gamma * max_next_q_value - current_q_value)
        self.update_q_value(hashed_canonical_current_state, canonical_move, updated_q_value)

    def is_terminal(self, state):
      return self.win(state) or self.tie(state) or self.game_over(state)
    def max_q_value(self, state, hashed_state):
        """
            Finds the action with the highest Q-value for a given state.

            Returns:
            - max_move: The action with the highest Q-value.
            - max_value: The Q-value of that action.
        """
        available_moves = self.find_available_moves(state)
        max_value = -math.inf
        max_move = available_moves[0]
        for move in available_moves:
            q_value = self.q_table.get((hashed_state, move), 0)
            if q_value > max_value:
                max_value = self.q_table.get((hashed_state, move), 0)
                max_move = move
        return [max_move, max_value]

    def find_available_moves(self, state):
        return [index for index, value in enumerate(state) if value == 0]

    def play(self):
        hashed_canonical_state, canonical_state, transformation_index = self.canonical_state_hash(
            self.board.get_state())
        if np.random.uniform(0, 1) <= self.exploration_rate:
            canonical_move = random.choice(self.find_available_moves(canonical_state))
            random_move = self.inverse_canonical_move(canonical_move, transformation_index)
            self.move(random_move)

        else:
            canonical_move = self.max_q_value(canonical_state, hashed_canonical_state)[0]
            original_move = self.inverse_canonical_move(canonical_move, transformation_index)
            self.move(original_move)

        new_state = self.board.get_state()
        hashed_canonical_new_state, canonical_new_state, new_transformation_index = self.canonical_state_hash(new_state)
        reward = self.reward(new_state)
        self.update_q_table(hashed_canonical_state, canonical_move, reward, canonical_new_state,
                            hashed_canonical_new_state)



# Training the agent with a random opponent:

In [18]:
num_episodes = 10000
epsilon = 0.1
gamma = 0.6
learning_rate = 0.9


def random_opponent(board):
    available_moves = [i for i, v in enumerate(board.get_state()) if v == 0]
    return random.choice(available_moves)
def train(epsilon, gamma, learning_rate, num_episodes):
    agent = Agent(player=1, epsilon=epsilon, gamma=gamma, learning_rate=learning_rate)
    for episode in range(num_episodes):
        # Reset the board for a new game
        agent.board = Board()
        agent.possible_moves = list(range(25))

        done = False
        turn = 1  # Agent starts first

        while not done:
            # Agent's turn
            if turn == agent.player:
                agent.play()
                reward = agent.reward(agent.board.get_state())
                done = reward == 5 or reward == -5 or agent.tie(agent.board.get_state())
            # Opponent's turn
            else:
                opponent_move = random_opponent(agent.board)
                agent.board.update_state(-agent.player, opponent_move)
                agent.possible_moves.remove(opponent_move)
                reward = agent.reward(agent.board.get_state())
                done = reward == 5 or reward == -5 or agent.tie(agent.board.get_state())

            turn = -turn
    return agent


trained_agent = train(epsilon, gamma, learning_rate, num_episodes)

# Testing the agent against a random opponent

In [19]:
def test_agent(agent, num_games=1000):
    wins = 0
    losses = 0
    ties = 0

    for _ in range(num_games):
        agent.board = Board()
        agent.possible_moves = list(range(25))
        done = False
        turn = 1

        while not done:
            # Agent's turn
            if turn == agent.player:
                state = agent.board.get_state()
                hashed_state, canonical_state, transformation_index = agent.canonical_state_hash(state)
                best_move = agent.max_q_value(canonical_state, hashed_state)[0]
                original_move = agent.inverse_canonical_move(best_move, transformation_index)
                agent.move(original_move)

                # Check game state after agent's move
                reward = agent.reward(agent.board.get_state())
                if reward == 5:
                    wins += 1
                elif reward == -5:
                    losses += 1
                elif reward == -0.2:
                  ties += 1
                done = reward == 5 or reward == -5 or agent.tie(agent.board.get_state())

            # Opponent's turn
            else:
                opponent_move = random_opponent(agent.board)
                agent.board.update_state(-agent.player, opponent_move)
                agent.possible_moves.remove(opponent_move)

                # Check game state after opponent's move
                reward = agent.reward(agent.board.get_state())
                if reward == 5:
                    losses += 1
                elif reward == -5:
                    wins += 1
                elif reward == -0.2:
                    ties += 1
                done = reward == 5 or reward == -5 or agent.tie(agent.board.get_state())

            turn = -turn  # Switch turn

    print(f"Out of {num_games} games:")
    print(f"Wins: {wins}")
    print(f"Gameovers: {losses}")
    print(f"Ties: {ties}")
    return wins, losses, ties
test_agent(trained_agent)

Out of 1000 games:
Wins: 568
Gameovers: 0
Ties: 432


(568, 0, 432)

# Finding the best hyperparameters

In [9]:
import pandas as pd

epsilon_values = [0.001, 0.01, 0.1, 0.5]
gamma_values = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]
learning_rate_values = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7, 0.8, 0.9]

num_episodes = 1000

results = []

for epsilon in epsilon_values:
    for gamma in gamma_values:
        for learning_rate in learning_rate_values:
            trained_agent = train(epsilon, gamma, learning_rate, num_episodes)
            wins, game_overs, ties = test_agent(trained_agent)
            results.append({
                'epsilon': epsilon,
                'gamma': gamma,
                'learning_rate': learning_rate,
                'wins': wins,
                'win_rate': wins / num_episodes
            })

results_df = pd.DataFrame(results)
print(results_df)

best_params = results_df.loc[results_df['win_rate'].idxmax()]
print("Best parameters found:")
print(best_params)


Out of 1000 games:
Wins: 529
Gameovers: 0
Ties: 471
Out of 1000 games:
Wins: 518
Gameovers: 0
Ties: 482
Out of 1000 games:
Wins: 518
Gameovers: 0
Ties: 482
Out of 1000 games:
Wins: 545
Gameovers: 0
Ties: 455
Out of 1000 games:
Wins: 540
Gameovers: 0
Ties: 460
Out of 1000 games:
Wins: 554
Gameovers: 0
Ties: 446
Out of 1000 games:
Wins: 558
Gameovers: 0
Ties: 442
Out of 1000 games:
Wins: 521
Gameovers: 0
Ties: 479
Out of 1000 games:
Wins: 590
Gameovers: 0
Ties: 410
Out of 1000 games:
Wins: 542
Gameovers: 0
Ties: 458
Out of 1000 games:
Wins: 552
Gameovers: 0
Ties: 448
Out of 1000 games:
Wins: 587
Gameovers: 0
Ties: 413
Out of 1000 games:
Wins: 552
Gameovers: 0
Ties: 448
Out of 1000 games:
Wins: 504
Gameovers: 0
Ties: 496
Out of 1000 games:
Wins: 537
Gameovers: 0
Ties: 463
Out of 1000 games:
Wins: 562
Gameovers: 0
Ties: 438
Out of 1000 games:
Wins: 542
Gameovers: 0
Ties: 458
Out of 1000 games:
Wins: 588
Gameovers: 0
Ties: 412
Out of 1000 games:
Wins: 583
Gameovers: 0
Ties: 417
Out of 1000 